# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print(f"{metadata_obj.name}: {metadata_obj.description}")
print(f"Version: {metadata_obj.version}")
print(f"License: {metadata_obj.license}")
print(f"Keywords: {metadata_obj.keywords}")

## 2. Data Overview
Review available record sets and their IDs. Explore the fields within each record set using their `@id`s.

In [ ]:
# Get all available RecordSets by @id
record_sets = []
for recordset in dataset.record_sets:
    print(f"RecordSet @id: {recordset['@id']}, name: {recordset.get('name', '(no name)')}")
    record_sets.append(recordset['@id'])

# Explore fields/columns in each RecordSet by their @id
for record_set_id in record_sets:
    print(f"--- Fields and columns for RecordSet {record_set_id} ---")
    rs = dataset.get_record_set(record_set_id)
    if rs:
        for field in rs.get('field', []):
            fid = field.get('@id', '(no id)')
            fname = field.get('name', '(no name)')
            print(f"Field @id: {fid}, name: {fname}")
            if 'column' in field:
                for col in field['column']:
                    print(f"  Column @id: {col.get('@id', '(no id)')}, name: {col.get('name', '(no name)')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. You must reference each record set and field by its `@id`.

In [ ]:
# Extract data from each record set
# Use RecordSet @id from the overview
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        print(f"Loaded {len(records)} records from {record_set_id}")
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization and grouping. All fields and columns must be referenced by their `@id`.

In [ ]:
# Choose a RecordSet with loaded records (change `example_record_set_id` as appropriate)
# Example: use the first available RecordSet
example_record_set_id = record_sets[0] if record_sets else None
if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]

    # List numeric fields based on column data types
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields (@id): {numeric_fields}")

    if numeric_fields:
        # Use first numeric field @id for analysis
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()  # Use mean as an example threshold

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group by a categorical field if available
        cat_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        group_field_id = cat_fields[0] if cat_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the loaded DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_record_set_id in dataframes and numeric_fields:
    df = dataframes[example_record_set_id]
    numeric_field_id = numeric_fields[0]

    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if both numeric and categorical field are present
    if cat_fields:
        group_field_id = cat_fields[0]
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed regression outputs for predictors related to knowledge adoption in rangeland management, including socio-demographics and intervention outcomes.
- Filtering and normalization demonstrate potential for identifying subgroups and normalization of numeric outcomes.
- Grouping by categorical attributes (referenced via `@id`) helps reveal potential patterns in adoption predictors.
- Visualization highlights the distribution and variation present in key numeric fields.

Further analysis can extend to more sophisticated modeling or policy interpretation based on the FAIR^2 dataset structure.